# Gemma 4 12B — SRD-4 vs Standard Q4_0 Comparison

Packs `google/gemma-4-12b-it-assistant` (BF16) into a signed SRD-4 `.axm` container,
extracts to GGUF Q4_K_M, then benchmarks it head-to-head against a
community Q4_0 GGUF on WikiText-2 PPL.

> **HF model ID note:** Verify the exact model ID on HuggingFace Hub before running.
> `google/gemma-4-12b-it-assistant` is used here; adjust `MODEL_ID` in Cell 2 if needed
> (e.g. `google/gemma-4-12b-it-assistant` or similar).
> Gemma 4 models require accepting the license on HuggingFace — set `HF_TOKEN`.

**Why this comparison matters**

| Method | bpw | Training change? | Key property |
|---|---|---|---|
| SRD Q4_K_M (this notebook) | ~4.85 | ✗ Post-training only | Stochastic residual dithering + HMAC governance |
| Standard Q4_0 (community) | ~4.0 | ✗ Post-training only | Standard llama.cpp quantization, lower bpw |

**What this notebook produces**

| Output | Size (est.) | Use |
|---|---|---|
| `gemma4_12b_srd4.axm` | ~6.5 GB | Signed SRD-4 container (~4.5 bpw) |
| `gemma4_12b_srd4_q4km.gguf` | ~7.5 GB | llama.cpp / PocketPal / llama-server |
| `gemma4_12b_std_q4_0.gguf` | ~6.5 GB | Community Q4_0 GGUF (bartowski) |
| `gemma4_12b_met_sidecar.json` | ~10 KB | MET hydration budget reference |
| PPL comparison table | — | SRD vs standard Q4_0 on WikiText-2 |

> **Disk note:** The BF16 model (~24 GB) + Q4_K_M GGUF (~7.5 GB) + Q4_0 GGUF (~6.5 GB)
> totals ~38 GB. Cell 4 converts BF16 → Q4_K_M directly (no intermediate F16 GGUF)
> to save ~24 GB of disk space. Colab Pro with A100 (~200 GB disk) handles this fine.

**Architecture (Gemma 4 12B — read from config.json at runtime)**

| Field | Expected (verify from config.json) |
|---|---|
| Parameters | ~12 B |
| vocab_size | 262,144 (Gemma family) |
| hidden_size | ~3,072 |
| num_layers | ~28 |
| num_attention_heads | ~16 |
| num_kv_heads | ~8 (GQA) |
| intermediate_size | ~24,576 |
| MLP style | GeGLU (same as Gemma 3) |

> Architecture values above are estimates; Cell 2 prints the actual values from config.json.

**Runtime:** A100 40 GB strongly recommended.
- T4 (15 GB): **OOM during Cell 3** (BF16 pack needs ~26 GB VRAM) — use `--device cpu` (very slow, ~3 hr).
- A100 40 GB: Cell 3 ~40 min, full run ~2–2.5 hr.
- A100 80 GB / H100: Cell 3 ~20 min.

> **Patent pending — Orivael Inc.** The SRD container format and signing protocol are the subject of pending patent claims.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — GPU check · clone axiom · install deps · persist AXIOM_MASTER_KEY
#           clone + cmake-build llama.cpp (GGML_CUDA=ON)  (~10-15 min)
# ══════════════════════════════════════════════════════════════════════════════
import os, secrets, subprocess, sys, time
from pathlib import Path

AXIOM_DIR  = Path('/content/axiom')
OUTPUT_DIR = Path('/content/axiom_output')
LLAMACPP   = Path('/content/llama.cpp')
BRANCH     = 'claude/srd-prototype-benchmark-JRtv1'
REPO_URL   = 'https://github.com/orivael-dev/axiom.git'
KEY_FILE   = OUTPUT_DIR / 'axiom_master.key'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── GPU info ──────────────────────────────────────────────────────────────────
import torch
p = torch.cuda.get_device_properties(0)
vram_gb   = p.total_memory / 1024**3
cuda_arch = p.major * 10 + p.minor
print(f'GPU : {p.name}  {vram_gb:.1f} GB VRAM  SM {p.major}.{p.minor}  arch={cuda_arch}')
if vram_gb < 20:
    print(f'  ⚠  {vram_gb:.0f} GB VRAM — 12B BF16 pack needs ~26 GB.')
    print(f'     Cell 3 will pack on CPU (very slow, ~3 hr). A100 recommended.')
elif vram_gb < 40:
    print(f'  ⚠  {vram_gb:.0f} GB VRAM — marginal for 12B BF16. May need CPU offload.')
else:
    print(f'  ✓ {vram_gb:.0f} GB VRAM — 12B BF16 fits comfortably')

# ── Clone axiom ───────────────────────────────────────────────────────────────
if not AXIOM_DIR.is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    REPO_URL, str(AXIOM_DIR)], check=True)
    print(f'✓ axiom cloned  ({BRANCH})')
else:
    r = subprocess.run(['git', '-C', str(AXIOM_DIR), 'pull', '--ff-only'],
                       capture_output=True, text=True)
    print(f'✓ axiom  {r.stdout.strip() or "up to date"}')

sys.path.insert(0, str(AXIOM_DIR))

# ── Persist AXIOM_MASTER_KEY ──────────────────────────────────────────────────
if KEY_FILE.is_file() and not os.environ.get('AXIOM_MASTER_KEY'):
    os.environ['AXIOM_MASTER_KEY'] = KEY_FILE.read_text().strip()
    print('AXIOM_MASTER_KEY restored from session key file')
elif not os.environ.get('AXIOM_MASTER_KEY'):
    key = secrets.token_hex(32)
    os.environ['AXIOM_MASTER_KEY'] = key
    KEY_FILE.write_text(key)
    print(f'AXIOM_MASTER_KEY generated → {KEY_FILE}')
    print('  ⚠ back this up — same key required to verify the .axm later')
else:
    print('AXIOM_MASTER_KEY already set')

# ── Install deps ──────────────────────────────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'datasets', 'huggingface_hub', 'tqdm',
    'accelerate', 'gguf>=0.10', 'sentencepiece'], check=True)
print('✓ deps installed')

# ── Build llama.cpp ───────────────────────────────────────────────────────────
if not (LLAMACPP / 'build' / 'bin' / 'llama-cli').is_file():
    print('Building llama.cpp...  (~10 min on A100)')
    t0 = time.time()
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggerganov/llama.cpp', str(LLAMACPP)], check=True)
    subprocess.run(
        ['cmake', '-B', 'build', '-DGGML_CUDA=ON',
         f'-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}'],
        cwd=LLAMACPP, check=True, capture_output=True)
    subprocess.run(
        ['cmake', '--build', 'build', '--config', 'Release', '-j4'],
        cwd=LLAMACPP, check=True, capture_output=True)
    print(f'✓ llama.cpp built  ({time.time()-t0:.0f} s)')
else:
    print('✓ llama.cpp already built')

LLAMA_CLI   = LLAMACPP / 'build' / 'bin' / 'llama-cli'
LLAMA_PPL   = LLAMACPP / 'build' / 'bin' / 'llama-perplexity'
LLAMA_QUANT = LLAMACPP / 'build' / 'bin' / 'llama-quantize'
LLAMA_BENCH = LLAMACPP / 'build' / 'bin' / 'llama-bench'
CONVERT     = next(LLAMACPP.glob('convert*.py'), None)
print(f'✓ ready  llama-cli={LLAMA_CLI.exists()}  llama-perplexity={LLAMA_PPL.exists()}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1b — Pre-download WikiText-2 test set
#
# Run immediately after Cell 1. Cells 6 and 7 skip the download if cached.
# Time: <1 min  |  Size: ~1.1 MB
# ══════════════════════════════════════════════════════════════════════════════
from pathlib import Path

OUTPUT_DIR = Path('/content/axiom_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WT2_PATH = OUTPUT_DIR / 'wikitext2_test.txt'

if WT2_PATH.exists():
    print(f'✓ wikitext-2 already cached  ({WT2_PATH.stat().st_size/1024:.0f} KB)  — nothing to do')
else:
    print('Downloading WikiText-2 test set from HuggingFace...')
    from datasets import load_dataset
    ds   = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
    text = '\n'.join(ds['text'])
    WT2_PATH.write_text(text, encoding='utf-8')
    kb   = WT2_PATH.stat().st_size / 1024
    print(f'✓ saved  {kb:.0f} KB  →  {WT2_PATH}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Download google/gemma-4-12b-it-assistant from HuggingFace
#
# ⚠ Gemma 4 requires accepting the license on HuggingFace Hub.
#   Set HF_TOKEN before running:
#     import os; os.environ['HF_TOKEN'] = 'hf_...'
#
# Adjust MODEL_ID if the exact HF repo name differs
# (e.g. 'google/gemma-4-12b-it-assistant').
#
# Size: ~24 GB (BF16 safetensors).  Time: 15-30 min on Colab.
# ══════════════════════════════════════════════════════════════════════════════
import json, os, time
from pathlib import Path
from huggingface_hub import snapshot_download

# ── Set HF_TOKEN if needed ────────────────────────────────────────────────────
# Uncomment and set your token if the model is gated:
# os.environ['HF_TOKEN'] = 'hf_your_token_here'

MODEL_ID  = 'google/gemma-4-12b-it-assistant'   # adjust if needed
MODEL_DIR = OUTPUT_DIR / 'gemma4_12b_hf'

if MODEL_DIR.is_dir() and any(MODEL_DIR.glob('*.safetensors')):
    print(f'✓ model already downloaded  →  {MODEL_DIR}')
else:
    print(f'Downloading {MODEL_ID} ...  (~24 GB BF16, 15-30 min)')
    t0 = time.time()
    snapshot_download(
        repo_id   = MODEL_ID,
        local_dir = str(MODEL_DIR),
        token     = os.environ.get('HF_TOKEN'),
        ignore_patterns = ['*.ot', 'flax_model*', 'tf_model*', 'rust_model*'],
    )
    elapsed = time.time() - t0
    files   = list(MODEL_DIR.glob('*.safetensors'))
    total_gb = sum(f.stat().st_size for f in files) / 1024**3
    print(f'✓ downloaded  {len(files)} shard(s)  {total_gb:.2f} GB  ({elapsed:.0f} s)')

# Architecture check
cfg = json.loads((MODEL_DIR / 'config.json').read_text())
print(f'\nArchitecture:')
print(f'  model_type         : {cfg.get("model_type")}')
print(f'  vocab_size         : {cfg.get("vocab_size"):,}')
print(f'  hidden_size        : {cfg.get("hidden_size")}')
print(f'  num_hidden_layers  : {cfg.get("num_hidden_layers")}')
print(f'  num_attention_heads: {cfg.get("num_attention_heads")}')
print(f'  num_kv_heads       : {cfg.get("num_key_value_heads")}')
print(f'  intermediate_size  : {cfg.get("intermediate_size")}')
print(f'  max_pos_embeddings : {cfg.get("max_position_embeddings"):,}')
hidden = cfg.get("hidden_size", 0)
heads  = cfg.get("num_attention_heads", 1)
print(f'  head_dim           : {cfg.get("head_dim", hidden // heads)}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — SRD-4 pack → gemma4_12b_srd4.axm
#
# SRD-4 = W4 base quantization, ~4.5 bpw, REAL-PACKED on disk.
# Signs every weight shard with HMAC-SHA256 — fingerprint is a public
# commitment to the exact packed weights.
#
# VRAM:  A100 40 GB packs entirely on GPU (~40 min).
#        T4 / L4 (<20 GB): automatically uses CPU offload (~3 hr).
#
# Time: ~40 min on A100  |  ~3 hr on T4 (CPU offload)
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys, time
from pathlib import Path

AXM_PATH    = OUTPUT_DIR / 'gemma4_12b_srd4.axm'
PACK_SCRIPT = AXIOM_DIR / 'research' / 'quant' / 'pack_to_axm.py'

if AXM_PATH.exists():
    print(f'✓ .axm already exists  ({AXM_PATH.stat().st_size/1024**3:.2f} GB)  — skipping pack')
else:
    import torch
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    use_cpu = vram_gb < 22
    device  = 'cpu' if use_cpu else 'cuda'
    if use_cpu:
        print(f'  ℹ {vram_gb:.0f} GB VRAM — packing on CPU (slow but safe)')

    print(f'Packing {MODEL_ID} → SRD-4 .axm  (device={device}) ...')
    t0  = time.time()
    cmd = [
        sys.executable, str(PACK_SCRIPT),
        '--model',      str(MODEL_DIR),
        '--output',     str(AXM_PATH),
        '--srd4',
        '--real-pack',
        '--group-size', '64',
    ]
    if use_cpu:
        cmd += ['--device', 'cpu']
    r = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0
    if r.returncode != 0:
        print('STDERR:', r.stderr[-2000:])
        raise RuntimeError('pack_to_axm.py failed')
    print(f'✓ packed  ({elapsed/60:.1f} min)')

axm_gb    = AXM_PATH.stat().st_size / 1024**3
bf16_gb   = sum(f.stat().st_size for f in MODEL_DIR.glob('*.safetensors')) / 1024**3
print(f'  .axm size : {axm_gb:.2f} GB')
print(f'  vs BF16   : {bf16_gb:.1f} GB  →  {100*(1 - axm_gb/bf16_gb):.0f}% smaller')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Verify .axm proof ledger + convert → GGUF Q4_K_M
#
# Step 1: axm_cli.py verify — checks every HMAC-SHA256 proof shard
# Step 2: convert_hf_to_gguf.py MODEL_DIR → Q8_0 GGUF (intermediate, ~12 GB)
#         Using Q8_0 instead of F16 saves ~12 GB disk vs the F16 route.
# Step 3: llama-quantize Q8_0 → Q4_K_M (~7.5 GB final)
#         Q8_0 intermediate is deleted after quantization to free disk space.
#
# Time: ~10 min on A100
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys, time
from pathlib import Path

AXM_CLI    = AXIOM_DIR / 'axm_cli.py'
GGUF_Q8    = OUTPUT_DIR / 'gemma4_12b_q8_0.gguf'    # intermediate, deleted after
GGUF_SRD   = OUTPUT_DIR / 'gemma4_12b_srd4_q4km.gguf'

# ── Step 1: Verify .axm ───────────────────────────────────────────────────────
print('Verifying .axm proof chain...')
r = subprocess.run(
    [sys.executable, str(AXM_CLI), 'verify', str(AXM_PATH)],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('VERIFY FAILED:', (r.stdout + r.stderr)[-1000:])
    raise RuntimeError('Proof verification failed')
try:
    vdata  = json.loads(r.stdout)
    fp     = vdata.get('fingerprint', 'n/a')
    proofs = vdata.get('proofs_checked', 'n/a')
except Exception:
    fp, proofs = 'see output', 'see output'
print(f'  ✓ verified  fingerprint={fp}  proofs={proofs}')

# ── Step 2: MODEL_DIR → Q8_0 GGUF (skip if Q4_K_M already done) ──────────────
if GGUF_SRD.exists():
    print(f'\n✓ Q4_K_M GGUF already exists  ({GGUF_SRD.stat().st_size/1024**2:.0f} MB)')
else:
    if not GGUF_Q8.exists():
        print(f'\nConverting {MODEL_ID} → Q8_0 GGUF  (~10 min, saves disk vs F16)...')
        t0 = time.time()
        r  = subprocess.run(
            [sys.executable, str(CONVERT),
             str(MODEL_DIR), '--outfile', str(GGUF_Q8), '--outtype', 'q8_0'],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print('OUTPUT:', (r.stdout + r.stderr)[-3000:])
            raise RuntimeError('convert_hf_to_gguf.py failed')
        print(f'  ✓ Q8_0 GGUF  ({GGUF_Q8.stat().st_size/1024**2:.0f} MB)  ({time.time()-t0:.0f} s)')
    else:
        print(f'\n✓ Q8_0 GGUF already exists  ({GGUF_Q8.stat().st_size/1024**2:.0f} MB)')

    # ── Step 3: Q8_0 → Q4_K_M ────────────────────────────────────────────────
    print(f'\nQuantizing Q8_0 → Q4_K_M...')
    t0 = time.time()
    r  = subprocess.run(
        [str(LLAMA_QUANT), str(GGUF_Q8), str(GGUF_SRD), 'Q4_K_M'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print('OUTPUT:', (r.stdout + r.stderr)[-2000:])
        raise RuntimeError('llama-quantize failed')
    print(f'  ✓ Q4_K_M  ({time.time()-t0:.0f} s)')

    # Delete Q8_0 intermediate to free ~12 GB
    if GGUF_Q8.exists():
        GGUF_Q8.unlink()
        print('  ✓ deleted Q8_0 intermediate (freed ~12 GB)')

gguf_mb = GGUF_SRD.stat().st_size / 1024**2
print(f'\n  SRD Q4_K_M : {gguf_mb:.0f} MB')
print(f'  fingerprint verified: {fp}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Generate MET RAM sidecar (Gemma 4 12B architecture)
#
# Reads architecture from config.json; no hardcoded values.
# Expected hydration budgets (Gemma 4 12B, ~Q4_K_M 4.85 bpw):
#   Embedding (F16, always pinned) : ~580 MB  (vocab=262144, h=3072)
#   INFORM (early layers only)     : ~2.8 GB
#   HARM (all layers)              : ~5.6 GB
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys
from pathlib import Path

SIDECAR_PATH = OUTPUT_DIR / 'gemma4_12b_met_sidecar.json'
ESTIMATOR    = AXIOM_DIR / 'research' / 'quant' / 'met_ram_estimator.py'

cfg    = json.loads((MODEL_DIR / 'config.json').read_text())
vocab  = cfg['vocab_size']
hidden = cfg['hidden_size']
layers = cfg['num_hidden_layers']
heads  = cfg['num_attention_heads']
kv     = cfg.get('num_key_value_heads', heads)
inter  = cfg['intermediate_size']

# Detect MLP style — Gemma 4 uses GeGLU (same family as Gemma 3)
act = cfg.get('hidden_act', '').lower()
mlp_style = 'geglu' if 'gelu' in act else 'swiglu'
print(f'  hidden_act={cfg.get("hidden_act")} → mlp_style={mlp_style}')

r = subprocess.run([
    sys.executable, str(ESTIMATOR),
    '--vocab-size',        str(vocab),
    '--hidden-size',       str(hidden),
    '--num-layers',        str(layers),
    '--num-heads',         str(heads),
    '--num-kv-heads',      str(kv),
    '--intermediate-size', str(inter),
    '--bpw',               '4.85',
    '--mlp-style',         mlp_style,
    '--model-id',          MODEL_ID,
    '--output',            str(SIDECAR_PATH),
], capture_output=True, text=True)

print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError(f'met_ram_estimator.py failed (rc={r.returncode})')

sd  = json.loads(SIDECAR_PATH.read_text())
hyd = sd['intent_ram_budget']
emb = sd['embedding_slot']['mb']
inform_mb   = hyd['INFORM']['total_mb']
harm_mb     = hyd['HARM']['total_mb']
savings_pct = 100 * (1 - inform_mb / harm_mb)

print(f'\n  Embedding (pinned F16) : {emb:.0f} MB')
print(f'  INFORM budget          : {inform_mb:.0f} MB  ({hyd["INFORM"]["ufs_load_ms"]:.0f} ms load)')
print(f'  HARM budget            : {harm_mb:.0f} MB  ({hyd["HARM"]["ufs_load_ms"]:.0f} ms load)')
print(f'  Savings INFORM vs HARM : {savings_pct:.1f} %')
print(f'  Sidecar saved → {SIDECAR_PATH}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — WikiText-2 PPL: SRD Q4_K_M
#
# Same settings as all other Axiom benchmarks:
#   stride 512, context 2048, 100 chunks
#
# 12B models typically score PPL 6-10 on WikiText-2.
# Time: ~20 min on A100  |  ~35 min on T4
# ══════════════════════════════════════════════════════════════════════════════
import json, re, subprocess, sys, time
from pathlib import Path

WT2_PATH = OUTPUT_DIR / 'wikitext2_test.txt'
if not WT2_PATH.exists():
    from datasets import load_dataset
    ds   = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
    WT2_PATH.write_text('\n'.join(ds['text']))
    print(f'✓ wikitext-2 saved')
else:
    print(f'✓ wikitext-2 already cached')

print('\nRunning PPL: SRD Q4_K_M  (~20 min on A100)...')
t0 = time.time()
r  = subprocess.run([
    str(LLAMA_PPL),
    '-m',             str(GGUF_SRD),
    '-f',             str(WT2_PATH),
    '--ctx-size',     '2048',
    '--ppl-stride',   '512',
    '--chunks',       '100',
    '--n-gpu-layers', '99',
], capture_output=True, text=True)
elapsed_srd = time.time() - t0

ppl_lines = [l for l in (r.stdout + r.stderr).splitlines()
             if 'Final estimate' in l or 'PPL' in l]
print(f'PPL output ({elapsed_srd/60:.1f} min):')
for l in ppl_lines:
    print(f'  {l.strip()}')

ppl_srd = None
m = re.search(r'PPL\s*=\s*([0-9]+\.[0-9]+)', r.stdout + r.stderr)
if m:
    ppl_srd = float(m.group(1))
    print(f'\n  SRD Q4_K_M PPL : {ppl_srd:.2f}')
    print(f'  12B models typically reach PPL 6-10 on WikiText-2 at Q4_K_M')

srd_result = {'ppl': ppl_srd, 'elapsed_min': round(elapsed_srd/60, 1)}
(OUTPUT_DIR / '_srd_ppl_tmp.json').write_text(json.dumps(srd_result))


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Download community Q4_0 GGUF + run WikiText-2 PPL
#
# Downloads bartowski/google-gemma-4-12b-it-assistant-GGUF Q4_0.
# Falls back to searching all available GGUF files if Q4_0 not found.
#
# Q4_0 bpw ≈ 4.0  vs  SRD Q4_K_M bpw ≈ 4.85
# Time: ~10 min download + ~20 min PPL on A100
# ══════════════════════════════════════════════════════════════════════════════
import json, os, re, subprocess, sys, time
from pathlib import Path
from huggingface_hub import hf_hub_download

STD_REPO_ID   = 'bartowski/google-gemma-4-12b-it-assistant-GGUF'
GGUF_STD_Q4_0 = OUTPUT_DIR / 'gemma4_12b_std_q4_0.gguf'

if GGUF_STD_Q4_0.exists():
    print(f'✓ Standard Q4_0 GGUF already present  ({GGUF_STD_Q4_0.stat().st_size/1024**2:.0f} MB)')
else:
    print(f'Downloading Q4_0 from {STD_REPO_ID} ...')
    t0 = time.time()
    try:
        from huggingface_hub import list_repo_files
        all_files  = list(list_repo_files(STD_REPO_ID, token=os.environ.get('HF_TOKEN')))
        gguf_files = [f for f in all_files if f.endswith('.gguf')]
        print(f'  Available: {gguf_files}')
        q4_files   = [f for f in gguf_files if 'Q4_0' in f.upper() and 'IQ' not in f.upper()]
        gguf_fname = q4_files[0] if q4_files else (gguf_files[0] if gguf_files else None)
        print(f'  Selected: {gguf_fname}')
    except Exception as e:
        print(f'  list_repo_files failed ({e}) — trying default filename')
        gguf_fname = 'google-gemma-4-12b-it-Q4_0.gguf'

    if gguf_fname is None:
        raise RuntimeError(f'No GGUF files found in {STD_REPO_ID}. '
                           f'Check the repo exists or set STD_REPO_ID manually.')

    local_path = hf_hub_download(
        repo_id   = STD_REPO_ID,
        filename  = gguf_fname,
        local_dir = str(OUTPUT_DIR),
        token     = os.environ.get('HF_TOKEN'),
    )
    Path(local_path).rename(GGUF_STD_Q4_0)
    print(f'✓ downloaded  {GGUF_STD_Q4_0.stat().st_size/1024**2:.0f} MB  ({time.time()-t0:.0f} s)')

# ── Run PPL ───────────────────────────────────────────────────────────────────
WT2_PATH = OUTPUT_DIR / 'wikitext2_test.txt'
if not WT2_PATH.exists():
    from datasets import load_dataset
    ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
    WT2_PATH.write_text('\n'.join(ds['text']))

print('\nRunning PPL: Standard Q4_0  (~20 min on A100)...')
t0 = time.time()
r  = subprocess.run([
    str(LLAMA_PPL),
    '-m',             str(GGUF_STD_Q4_0),
    '-f',             str(WT2_PATH),
    '--ctx-size',     '2048',
    '--ppl-stride',   '512',
    '--chunks',       '100',
    '--n-gpu-layers', '99',
], capture_output=True, text=True)
elapsed_std = time.time() - t0

ppl_lines = [l for l in (r.stdout + r.stderr).splitlines()
             if 'Final estimate' in l or 'PPL' in l]
print(f'PPL output ({elapsed_std/60:.1f} min):')
for l in ppl_lines:
    print(f'  {l.strip()}')

ppl_std = None
m = re.search(r'PPL\s*=\s*([0-9]+\.[0-9]+)', r.stdout + r.stderr)
if m:
    ppl_std = float(m.group(1))
    print(f'\n  Standard Q4_0 PPL : {ppl_std:.2f}')

std_result = {'ppl': ppl_std, 'elapsed_min': round(elapsed_std/60, 1)}
(OUTPUT_DIR / '_std_ppl_tmp.json').write_text(json.dumps(std_result))


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Side-by-side comparison + llama-bench speed + save results
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys, time
from pathlib import Path

OUTPUT_DIR    = Path('/content/axiom_output')
GGUF_SRD      = OUTPUT_DIR / 'gemma4_12b_srd4_q4km.gguf'
GGUF_STD_Q4_0 = OUTPUT_DIR / 'gemma4_12b_std_q4_0.gguf'
SIDECAR_PATH  = OUTPUT_DIR / 'gemma4_12b_met_sidecar.json'
LLAMACPP      = Path('/content/llama.cpp')
LLAMA_BENCH   = LLAMACPP / 'build' / 'bin' / 'llama-bench'

srd_tmp = json.loads((OUTPUT_DIR / '_srd_ppl_tmp.json').read_text()) \
          if (OUTPUT_DIR / '_srd_ppl_tmp.json').exists() else {}
std_tmp = json.loads((OUTPUT_DIR / '_std_ppl_tmp.json').read_text()) \
          if (OUTPUT_DIR / '_std_ppl_tmp.json').exists() else {}

ppl_srd = srd_tmp.get('ppl')
ppl_std = std_tmp.get('ppl')

srd_mb = GGUF_SRD.stat().st_size      / 1024**2 if GGUF_SRD.exists()      else 0
std_mb = GGUF_STD_Q4_0.stat().st_size / 1024**2 if GGUF_STD_Q4_0.exists() else 0

# ── llama-bench ───────────────────────────────────────────────────────────────
bench = {}
for label, path in [('SRD Q4_K_M', GGUF_SRD), ('Standard Q4_0', GGUF_STD_Q4_0)]:
    if not path.exists() or not LLAMA_BENCH.exists():
        bench[label] = {'tg_ts': None, 'pp_ts': None}; continue
    print(f'Running llama-bench: {label}...')
    r = subprocess.run([
        str(LLAMA_BENCH), '-m', str(path),
        '-p', '512', '-n', '128', '-r', '3',
        '--n-gpu-layers', '99', '--output', 'json',
    ], capture_output=True, text=True)
    try:
        bd  = json.loads(r.stdout)
        lst = bd if isinstance(bd, list) else bd.get('results', [])
        tg  = next((x['avg_ts'] for x in lst if x.get('n_gen', 0) > 0), None)
        pp  = next((x['avg_ts'] for x in lst if x.get('n_prompt', 0) > 0), None)
        bench[label] = {'tg_ts': round(tg, 1) if tg else None,
                         'pp_ts': round(pp, 1) if pp else None}
    except Exception as e:
        print(f'  bench error: {e}')
        bench[label] = {'tg_ts': None, 'pp_ts': None}

_s = bench.get('SRD Q4_K_M', {}); _q = bench.get('Standard Q4_0', {})

# ── Table ─────────────────────────────────────────────────────────────────────
print()
print('═' * 82)
print('  GEMMA 4 12B — SRD Q4_K_M  vs  Standard Q4_0')
print('═' * 82)

rows = [
    ('Quant method',     'SRD-4 (post-training)',   'Standard Q4_0 (community)'),
    ('bpw',             '~4.85',                    '~4.0'),
    ('Size (MB)',        f'{srd_mb:.0f}' if srd_mb else 'n/a',
                         f'{std_mb:.0f}' if std_mb else 'n/a'),
    ('WikiText-2 PPL',   f'{ppl_srd:.2f}' if ppl_srd else 'run Cell 6',
                         f'{ppl_std:.2f}' if ppl_std else 'run Cell 7'),
    ('TG t/s (GPU)',     str(_s.get('tg_ts') or 'n/a'),  str(_q.get('tg_ts') or 'n/a')),
    ('PP t/s (GPU)',     str(_s.get('pp_ts') or 'n/a'),  str(_q.get('pp_ts') or 'n/a')),
    ('Training change?', 'No (drop-in)',             'No (drop-in)'),
    ('HMAC governance',  'Yes (fingerprinted)',      'No'),
    ('llama.cpp compat', 'Yes',                      'Yes'),
]

print(f'  {"Metric":<24} | {"SRD Q4_K_M":<30} | {"Standard Q4_0"}')
print(f'  {"-"*24}-+-{"-"*30}-+-{"-"*24}')
for metric, srd_val, std_val in rows:
    print(f'  {metric:<24} | {srd_val:<30} | {std_val}')

print()
if ppl_srd and ppl_std:
    delta = ppl_srd - ppl_std
    print(f'  PPL: SRD={ppl_srd:.2f}  |  Standard Q4_0={ppl_std:.2f}  |  Δ={delta:+.2f}')
    if delta < 0:
        print(f'  SRD wins by {abs(delta):.2f} PPL at 4.85 bpw vs Q4_0 at 4.0 bpw ✓')
    else:
        print(f'  Standard Q4_0 wins by {delta:.2f} PPL (lower bpw = expected)')
    if SIDECAR_PATH.exists():
        sd  = json.loads(SIDECAR_PATH.read_text())
        hyd = sd.get('intent_ram_budget', {})
        inf = hyd.get('INFORM', {}).get('total_mb', 0)
        hrm = hyd.get('HARM',   {}).get('total_mb', 0)
        if inf and hrm:
            print(f'\n  MET RAM: INFORM={inf:.0f} MB  |  HARM={hrm:.0f} MB  |  savings={100*(1-inf/hrm):.1f}%')

# ── Save JSON ─────────────────────────────────────────────────────────────────
results = {
    'model':     'google/gemma-4-12b-it-assistant',
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'srd_q4km': {
        'quant': 'SRD-4 → Q4_K_M', 'bpw': 4.85,
        'size_mb': round(srd_mb), 'ppl': ppl_srd,
        'tg_ts': _s.get('tg_ts'), 'pp_ts': _s.get('pp_ts'),
    },
    'std_q4_0': {
        'quant': 'Standard Q4_0 (community)', 'bpw': 4.0,
        'size_mb': round(std_mb), 'ppl': ppl_std,
        'tg_ts': _q.get('tg_ts'), 'pp_ts': _q.get('pp_ts'),
    },
    'benchmark_config': {
        'ppl_dataset': 'wikitext-2', 'ppl_stride': 512,
        'ppl_context': 2048, 'ppl_chunks': 100,
        'speed_pp': 512, 'speed_tg': 128, 'speed_reps': 3,
    },
}
RESULTS_PATH = OUTPUT_DIR / 'gemma4_12b_srd_vs_std.json'
RESULTS_PATH.write_text(json.dumps(results, indent=2))
print(f'\n  Results saved → {RESULTS_PATH}')
